# CW_05
# Revisiting the Double Slit Experiment

In this exercise, you will return to the double slit experiment that we saw at the beginning of the semester, but this time explore what happens when the slits are illuminated with incoherent light.

To do this, you will follow the methodology outlined in class which shows that one may model an incoherent source by sampling the propagation of many different copies of the source with random phase and summing up their intensities. To lighten the computational load of this, this workbook will be done in a 1D setting instead of 2D.

This workbook will walk you through creating this simulation, and then you will answer some questions about the output that you observe.

In [7]:
# - No modification required -

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from ipywidgets import interact, FloatSlider, IntSlider

# Global simulation parameters
wavelength = 633e-9 
k = 2 * np.pi / wavelength

N = 1024  # number of points
dx = 20e-6
L = N * dx

x = np.linspace(-L/2, L/2, N)
X = x  # 1D case

# Frequency coordinates
fx = np.fft.fftfreq(N, d=dx)
FX = fx

def angular_spectrum_propagation(field, z, n_slices=50):
    """
    Propagate a field using the angular spectrum method.
    
    If return_stack=True, returns intensity at intermediate slices.
    """
    F_field = np.fft.fft(field)
    
    kz = np.sqrt((k**2 - (2*np.pi*FX)**2).astype(complex))
    
    z_vals = np.linspace(0, z, n_slices)
    field_stack = np.zeros((n_slices, N), dtype=np.complex128)
    
    for i, zi in enumerate(z_vals):
        H = np.exp(1j * kz * zi)
        slice_field = np.fft.ifft(F_field * H)
        field_stack[i, :] = slice_field
    
    return field_stack, z_vals

# Part 1

In this first part, you will be responsible for creating an incoherent source for our double slit system. To do this, create a rectangular aperture of the passed size, then apply a random phase.

Once you have done this, use the provided visualization to sanity-check your output.

In [8]:
def generate_incoherent_source(aperture_size):
    """
    Generate a square aperture with random phase.
    """
    field = np.zeros_like(X, dtype=complex)
    
    aperture = np.abs(X) < (aperture_size / 2)
    
    random_phase = np.exp(1j * 2 * np.pi * np.random.rand(N))
    
    field[aperture] = random_phase[aperture]
    
    return field


# - No modification required -

def visualize_source(aperture_size):
    field = generate_incoherent_source(aperture_size)
    
    amplitude = np.abs(field)
    phase = np.angle(field)
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    
    axs[0].plot(X, amplitude)
    axs[0].set_title("Amplitude")
    
    axs[1].plot(X, phase)
    axs[1].set_title("Phase")
    
    plt.show()

interact(
    visualize_source,
    aperture_size=FloatSlider(min=0.1e-3, max=2e-3, step=0.1e-3, value=1e-3)
)

interactive(children=(FloatSlider(value=0.001, description='aperture_size', max=0.002, min=0.0001, step=0.0001…

<function __main__.visualize_source(aperture_size)>

# Part 2

Now that we have a source, we need our double slit. Following the same procedure as the previous double slit notebook, implement the function to create the slit.

Then use the visualization to check your function.

In [9]:
def double_slit(slit_width, slit_spacing):
    """
    Generate a double slit aperture.
    """
    slit1 = np.abs(X - slit_spacing/2) < (slit_width/2)
    slit2 = np.abs(X + slit_spacing/2) < (slit_width/2)
    
    aperture = slit1 | slit2
    
    return aperture.astype(float)


# - No modification required -

def visualize_slits(slit_width, slit_spacing):
    aperture = double_slit(slit_width, slit_spacing)
    
    plt.figure(figsize=(8, 3))
    plt.plot(X, aperture)
    plt.title("Double Slit Aperture")
    plt.ylim(-0.1, 1.1)
    plt.show()

interact(
    visualize_slits,
    slit_width=FloatSlider(min=0.01e-3, max=0.5e-3, step=0.01e-3, value=0.1e-3),
    slit_spacing=FloatSlider(min=0.1e-3, max=1e-3, step=0.05e-3, value=0.5e-3)
)

interactive(children=(FloatSlider(value=0.0001, description='slit_width', max=0.0005, min=1e-05, step=1e-05), …

<function __main__.visualize_slits(slit_width, slit_spacing)>

# Part 3

Now that you have your source and your aperture, it is time to combine them to create the whole system.

First, implement the function propagate_system. This function should generate a random input field and propagate it through the system with the provided angular spectrum routine. Note that this routine provides a cross section and its z values in addition to the final output. You will have to combine the outputs from each propagation step and convert them into intensity to return the final intensity, the intensity cross-section, and the z values along the way.

After you have implemented this function, you should implement the incoherent_average function. This function will run the propagate_system function many times and average together the resulting intensities. The return values will be in the same format as propagate_system, but this time will be the averaged values.

After both of these functions are implemented, use the visualize_system function to see the cross-section and final intensity of your incoherently illuminated system.

Then move to part 4 to answer some questions about what you see.

In [10]:
def propagate_system(aperture_size, z1, slit_width, slit_spacing, z2):
    """
    Full system:
    source → propagate → double slit → propagate → detector
    """
    # Source
    field = generate_incoherent_source(aperture_size)
    
    # Propagate to slits
    stack1, z_vals1 = angular_spectrum_propagation(field, z1)
    field_at_slits = stack1[-1]
    
    # Apply slits
    aperture = double_slit(slit_width, slit_spacing)
    field_after_slits = field_at_slits * aperture

    stack2, z_vals2 = angular_spectrum_propagation(field_after_slits, z2)
    
    stack = np.vstack((stack1, stack2))
    z_vals = np.concatenate((z_vals1, z_vals2+z_vals1[-1]))
    
    intensity_stack = np.abs(stack)**2
    final_intensity = intensity_stack[-1]
    
    return final_intensity, intensity_stack, z_vals


In [11]:
def incoherent_average(n_samples, aperture_size, z1, slit_width, slit_spacing, z2):
    """
    Average many realizations.
    """
    avg_intensity = np.zeros(N)
    avg_stack = None
    
    for i in range(n_samples):
        intensity, stack, z_vals = propagate_system(
            aperture_size, z1, slit_width, slit_spacing, z2
        )
        
        avg_intensity += intensity
        
        if avg_stack is None:
            avg_stack = stack
        else:
            avg_stack += stack
    
    avg_intensity /= n_samples
    avg_stack /= n_samples
    
    return avg_intensity, avg_stack, z_vals

In [12]:
# - No modification required -

def visualize_system(aperture_size, z1, slit_width, slit_spacing, z2, n_samples):
    
    intensity, stack, z_vals = incoherent_average(
        n_samples, aperture_size, z1, slit_width, slit_spacing, z2
    )
    
    stack_safe = np.clip(stack, 1e-12, None)
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    Z, X_grid = np.meshgrid(z_vals, X, indexing='ij')    
    im = axs[0].pcolormesh(
        X_grid,
        Z,
        stack_safe,
        shading='auto',
        cmap='inferno',
        norm=LogNorm(vmin=1e-6, vmax=np.max(stack_safe))
    )
    
    axs[0].set_title("Propagation (Log Intensity)")
    axs[0].set_xlabel("x (m)")
    axs[0].set_ylabel("z (m)")
    
    cbar = fig.colorbar(im, ax=axs[0])
    cbar.set_label("Intensity (log scale)")
    
    # Final intensity
    axs[1].plot(X, intensity)
    axs[1].set_title("Final Intensity")
    axs[1].set_xlabel("x (m)")
    
    plt.tight_layout()
    plt.show()


interact(
    visualize_system,
    aperture_size=FloatSlider(min=0.1e-3, max=2e-3, step=0.1e-3, value=1e-3, readout_format='.3f'),
    z1=FloatSlider(min=0, max=0.5, step=0.05, value=0.25, readout_format='.4f'),
    slit_width=FloatSlider(min=20e-6, max=100e-6, step=20e-6, value=60e-6, readout_format='.6f'),
    slit_spacing=FloatSlider(min=100e-6, max=500e-6, step=100e-6, value=200e-6, readout_format='.6f'),
    z2=FloatSlider(min=0, max=0.5, step=0.05, value=0.25, readout_format='.2f'),
    n_samples=IntSlider(min=1, max=200, step=1, value=50)
)

interactive(children=(FloatSlider(value=0.001, description='aperture_size', max=0.002, min=0.0001, readout_for…

<function __main__.visualize_system(aperture_size, z1, slit_width, slit_spacing, z2, n_samples)>

# Part 4

Now that you have put together your whole simulation, it's time to investigate what you are seeing. Using the provided sliders to adjust system parameters, answer the following questions:

1) What parameters are most responsible for how similar the output of the system looks to what we would expect from coherent illumination? For each of those parameters, how should you change them to increase coherence? Can you provide an intuitive explanation of why this works?
2) What do you notice as the main differences in the system output between when the source light appears to be incoherent and when it appears to be coherent?
3) How does changing the number of times that we sample the system affect the quality of our outputs? When do we need to sample many times and when can we get away with just a few samples?

## Discussion
1) The size of the source light and its distance to the aperture are the main ways that we can control the coherence of the light that illuminates the double slit. Increasing the distance of the source increases its coherence, and decreasing its size increases its coherence as well. We can think of how this helps by understanding that light is more spatially coherent when it comes from a more similar set of angles. By decreasing the size of the source or by increase its distance, we decrease the possible angles that light that hits the double slit could be coming from.
2) When the source is nearly coherent, the output of the system seems to be approximately what we see for a perfectly coherent source. As the coherence of the source falls, the visibility of the fringes falls as well, and we move more towards just seeing a large envelope function with little definition.
3) For highly incoherent light we find that we need to sample the system many times to get an output that converges. If we do not sample it enough, we see behavior such as an asymmetric output which we know must be unphysical. When the light is highly coherent, we do not need to sample the system as much, and for perfectly coherent light we would only need to sample one time.